# Feature Engineering `accounts` Data

In [3]:
#importing relevant libraries
# in terminal: uv sync
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import re


In [4]:
#looading clean accounts dataset generated via aml_clean.ipynb
accounts = pd.read_csv('../../data/processed/aml_clean.csv')
accounts.head()


,date,time,from_bank,account,to_bank,account.1,amount_received,receiving_currency,amount_paid,payment_currency,...,s_bank_type,r_bank_name,r_bank_id,r_entity_id,r_entity_name,r_entity_type,r_country,r_latitude,r_longitude,r_bank_type
0,2022-09-01,00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,...,Commercial,National Bank of Laramie,10,800D232D0,Partnership #1,Partnership,USA,39.783730,-100.445882,Commercial
1,2022-09-01,00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,...,Cooperative,Arbor Savings Bank,1,800AA5D20,Corporation #1,Corporation,other,18.515756,-77.834797,Savings
2,2022-09-01,00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,...,Commercial,National Bank of Fort Wayne,3209,800FBB3A0,Partnership #3,Partnership,USA,39.783730,-100.445882,Commercial
3,2022-09-01,00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,...,Commercial,National Bank of the East,12,800C0EF20,Sole Proprietorship #1,Sole Proprietorship,USA,39.783730,-100.445882,Commercial
4,2022-09-01,00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,...,Commercial,National Bank of Laramie,10,800C3EC10,Partnership #4,Partnership,USA,39.783730,-100.445882,Commercial


# One Hot Encoding Catagories From `accounts` Data

In [5]:
#pulling payment features for encoding
payment_cat_features = ['payment_currency', 'receiving_currency', 's_country', 'r_country', 's_bank_type', 'r_bank_type', 'payment_format','s_entity_type', 'r_entity_type']

In [ ]:
#sampling for development purposes
accounts_sample = accounts.sample(n=50000, random_state=42)

#preparing the column transformer for encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), payment_cat_features)
    ],
    remainder='passthrough'
)

transformed_array = preprocessor.fit_transform(accounts_sample)
encoded_cols = preprocessor.named_transformers_['cat'].get_feature_names_out(payment_cat_features)
passthrough_cols = [col for col in accounts_sample.columns if col not in payment_cat_features]

#combining the encoded and passthrough columns
all_account_cols = list(encoded_cols) + passthrough_cols

df_transformed = pd.DataFrame(transformed_array, columns=all_account_cols)

In [8]:
print(df_transformed.shape) 
print(df_transformed.columns.tolist())

(50000, 149)
['payment_currency_Australian Dollar', 'payment_currency_Bitcoin', 'payment_currency_Brazil Real', 'payment_currency_Canadian Dollar', 'payment_currency_Euro', 'payment_currency_Mexican Peso', 'payment_currency_Ruble', 'payment_currency_Rupee', 'payment_currency_Saudi Riyal', 'payment_currency_Shekel', 'payment_currency_Swiss Franc', 'payment_currency_UK Pound', 'payment_currency_US Dollar', 'payment_currency_Yen', 'payment_currency_Yuan', 'receiving_currency_Australian Dollar', 'receiving_currency_Bitcoin', 'receiving_currency_Brazil Real', 'receiving_currency_Canadian Dollar', 'receiving_currency_Euro', 'receiving_currency_Mexican Peso', 'receiving_currency_Ruble', 'receiving_currency_Rupee', 'receiving_currency_Saudi Riyal', 'receiving_currency_Shekel', 'receiving_currency_Swiss Franc', 'receiving_currency_UK Pound', 'receiving_currency_US Dollar', 'receiving_currency_Yen', 'receiving_currency_Yuan', 's_country_Australia', 's_country_Austria', 's_country_Belgium', 's_co

# Creating Catagories From `accounts` Data

In [9]:
#sampling for development purposes
accounts_sample = accounts.sample(n=50000, random_state=42)

#amount in should be amount out, so we can create a ratio and difference feature to capture this relationship
#ratio close to 1 we can flag as suspicious?
#difference close to 0 we can flag as suspicious?
accounts['amount_ratio'] = accounts['amount_received'] / accounts['amount_paid']
accounts['amount_diff'] = (accounts['amount_paid'] - accounts['amount_received']).abs()


#international transaction flag. International transactions may be more likely to be suspicious, so we can create a binary feature to capture this.
accounts['is_international'] = (accounts['s_country'] != accounts['r_country']).astype(int) 